# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end workflow for loading and exploring the FAIR^2 protein mass spectrometry dataset using the `mlcroissant` library and referencing all entities by their `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# View summary dataset metadata
metadata = dataset.metadata
print("Dataset Name:", getattr(metadata, 'name', None))
print("Description:", getattr(metadata, 'description', None))
print("Identifier:", getattr(metadata, 'identifier', None))
print("License:", getattr(metadata, 'license', None))
print("Version:", getattr(metadata, 'version', None))

## 2. Data Overview
Review available `RecordSet` entities, fields, and their IDs with the Croissant API. All entities are referenced by their `@id` field.

A Croissant schema may include multiple record sets (tables).

In [ ]:
# List all RecordSets with their @id and schema.name
print("Listing all record sets and their fields:\n")
record_sets = dataset.metadata.record_sets
record_set_ids = []
for rs in record_sets:
    rs_id = rs.id  # This is the @id field
    record_set_ids.append(rs_id)
    print(f"RecordSet @id: {rs_id}, name: {getattr(rs, 'name', None)}")
    # List fields with @id and name for each RecordSet
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, name: {getattr(field, 'name', None)}, dataType: {getattr(field, 'data_type', None)}")
    print("")
# Display the first record from the first record set as an example
if record_set_ids:
    rs_id = record_set_ids[0]
    records_iter = dataset.records(record_set=rs_id)
    try:
        first_record = next(records_iter)
        print(f"Example record from record set @id {rs_id}:")
        for k, v in first_record.items():
            print(f"  {k}: {v}")
    except StopIteration:
        print('No records found for this record set.')
else:
    print('No record sets available in this dataset.')

## 3. Data Extraction
Extract all data from each record set into pandas DataFrames for analysis. All record sets and fields are referenced via their `@id` as shown above.

In [ ]:
# Extract all records for each RecordSet into a dictionary of DataFrames
dataframes = {}
for rs_id in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"DataFrame for RecordSet @id '{rs_id}': columns: {df.columns.tolist()}")
    display(df.head())

# For further steps, choose the main RecordSet (typically the first one, but users may select any by @id)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Using main record set: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
else:
    main_df = pd.DataFrame()
    print('No record set found. Check dataset schema.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—e.g., filtering records based on a threshold, normalizing a numeric field, and grouping. All references use `@id` values.

**CHANGE the variables below (`numeric_field_id`, `group_field_id`) to appropriate `@id` values shown in the previous record set overview output.**

In [ ]:
# Replace these with actual field @id values from the above RecordSet introspection
# For example: numeric_field_id = 'cr:MW_kDa' for molecular weight; group_field_id = 'cr:sample_condition'
numeric_field_id = None # e.g., 'cr:MW_kDa'
group_field_id = None   # e.g., 'cr:sample_condition'

try:
    if main_record_set_id and not main_df.empty:
        if numeric_field_id is None:
            print('Please set numeric_field_id to the @id of a numeric field.')
        else:
            # Basic numeric filter and transformation example
            threshold = 10
            filtered_df = main_df[main_df[numeric_field_id].astype(float) > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold} (using @id reference):")
            display(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
            ) / filtered_df[numeric_field_id].astype(float).std()
            print(f"Normalized {numeric_field_id} (z-score) for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Grouping
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(f"Grouped data by {group_field_id} (using @id reference):")
                display(grouped_df)
            else:
                print("Set group_field_id to a categorical @id for grouping, or verify the chosen field exists.")
    else:
        print('No main DataFrame to analyze. Please complete record extraction above.')
except Exception as e:
    print('Error during EDA:', e)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All field references should use `@id`.

**CHANGE plot_field_x and plot_field_y below to @id values from the RecordSet columns.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Set these to actual @id fields for plotting
plot_field_x = None # e.g., 'cr:MW_kDa'
plot_field_y = None # e.g., 'cr:Normalized_Abundance'

if main_record_set_id and not main_df.empty and plot_field_x and plot_field_y and plot_field_x in main_df.columns and plot_field_y in main_df.columns:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=main_df, x=plot_field_x, y=plot_field_y)
    plt.title(f"Scatter plot of {plot_field_x} vs {plot_field_y} (by @id)")
    plt.xlabel(plot_field_x)
    plt.ylabel(plot_field_y)
    plt.show()
else:
    print("Set plot_field_x and plot_field_y to @id values of numeric columns in the main DataFrame and rerun.")

## 6. Conclusion
This notebook demonstrated loading and exploring a FAIR^2 Croissant-compliant dataset using the `mlcroissant` library and strictly referencing all entities and fields by their `@id` as defined in the Croissant schema.

Key steps:
- Loaded metadata and listed available `RecordSet` and `Field` entities (`@id`).
- Extracted record data into pandas DataFrames by `RecordSet` `@id`.
- Showed templates for EDA and visualization using field `@id` references.

**To finalize your analysis:**
- Inspect field `@id` values for the main record set.
- Assign those IDs to `numeric_field_id`, `group_field_id`, `plot_field_x`, and `plot_field_y` as needed.
- Adapt the provided code for your analytical needs.